In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# PARAMETERS CELL
model = ""        # "" = build all active models
batch_id = ""

StatementMeta(, aac688c7-c287-47b8-a0cb-d7136347b30a, 7, Finished, Available, Finished, False)

In [3]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

batch_id = batch_id or str(uuid.uuid4())

q = spark.table("meta_gold_models").filter(F.col("is_active") == 1)
if model:
    q = q.filter(F.col("model") == model)
cfgs = q.collect()
if not cfgs:
    raise ValueError(f"No active gold model found for '{model or 'ALL'}'")

failures = []

for cfg in cfgs:
    start_ts = datetime.utcnow()
    status, rows_written, error_msg = "SUCCESS", None, None
    try:
        df = spark.sql(cfg.sql_text)
        (df.write.format("delta").mode("overwrite")
           .option("overwriteSchema", "true")
           .saveAsTable(cfg.target_table))
        rows_written = spark.table(cfg.target_table).count()
        print(f"[{cfg.model}] wrote {rows_written} rows to {cfg.target_table}")
    except Exception as e:
        status, error_msg = "FAILED", str(e)[:4000]
        failures.append(cfg.model)
        print(f"[{cfg.model}] FAILED: {error_msg}")
    finally:
        spark.createDataFrame(
            [(batch_id, cfg.model, "gold", status, rows_written, start_ts, datetime.utcnow(), error_msg)],
            "batch_id string, entity string, layer string, status string, rows_written long, "
            "start_ts timestamp, end_ts timestamp, error_msg string"
        ).write.format("delta").mode("append").saveAsTable("meta_run_log")

if failures:
    raise Exception(f"Gold models failed: {failures}")

StatementMeta(, aac688c7-c287-47b8-a0cb-d7136347b30a, 8, Finished, Available, Finished, False)

/tmp/ipykernel_7242/4277272414.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow()
/tmp/ipykernel_7242/4277272414.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  [(batch_id, cfg.model, "gold", status, rows_written, start_ts, datetime.utcnow(), error_msg)],


[dim_constructors] wrote 214 rows to gold.dim_constructors
[dim_drivers] wrote 874 rows to gold.dim_drivers
[dim_races] wrote 1173 rows to gold.dim_races


StatementMeta(, aac688c7-c287-47b8-a0cb-d7136347b30a, 9, Finished, Available, Finished, False)

dim_constructors 214 vs silver 214
dim_drivers 874 vs silver 874
dim_races 1173 vs silver 1173
fact 26279 vs 26279


SynapseWidget(Synapse.DataFrame, 874c947c-192c-49c0-aa33-e9619f283267)